# iPSC-EC day 60 — endothelial vs fibroblast marker violins

Per-gene violin plots of six endothelial and six fibroblast markers across the two
day-60 iPSC-EC media conditions (5% FBS gfp vs. VEGF+SB+3% FBS), three biological
replicates each. Violins pool the replicates per condition; individual replicates are
overlaid as dots and the inner box shows the median and quartiles.

**Inputs** (all three required):
- `all_samples_old_and_new.csv` — batch 1/2 Cuffdiff FPKM matrix
- `IPSC_DAY60_batch3_expression_all_genes.csv` — batch 3 day-60 Cuffdiff FPKM matrix
- `IPSC_EC_highly_variable_genes.csv` — fixed highly-variable gene list

**Pipeline**: concatenate the batch 1/2 and batch 3 matrices -> drop genes detected in
fewer than 3 samples -> `normalize_total` to 10,000 per sample -> `log1p` -> restrict to
the supplied highly-variable gene list -> select the marker panel -> pool the three
replicates per condition.

Unlike Figures 1E/2D/4A this notebook does not use log2/log10(FPKM+1): it applies
per-sample total-count normalization followed by `log1p`, so the y-axis is a log-normalized
expression value. The batch 1/2 matrix is required even though only day-60 samples are
plotted, because the per-sample normalization total is computed over the genes retained
across all samples.

**Statistics**: brackets report a two-sided Welch t-test on n = 3 vs. n = 3 biological
replicates. At this sample size only *CD34* reaches p < 0.05 (p = 0.036) and no gene
survives multiple-testing correction across the twelve markers, so the annotations are
descriptive. Set `SHOW_STATS = False` to omit them.

The replicate dots are jittered with the global NumPy RNG; `RNG` is fixed so repeat runs
are byte-identical.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import anndata as ad
from scipy import stats
from pathlib import Path

RNG = 0
SHOW_STATS = True

DATA = Path("..")            # directory holding the three input matrices
OUT = Path("figures"); OUT.mkdir(exist_ok=True)

ALL_CSV = DATA / "all_samples_old_and_new.csv"
D60_CSV = DATA / "IPSC_DAY60_batch3_expression_all_genes.csv"
HVG_CSV = DATA / "IPSC_EC_highly_variable_genes.csv"

In [ ]:
# batch 1/2 matrix: keep only the day-0 iPSC-EC control columns, rename to conditions
batch2 = {
    "sample_1": "TELO HAEC+ctrl", "sample_2": "TELO HAEC+SB",
    "sample_3": "TELO HAEC+SB+cAMP", "sample_4": "TELO HAEC+ctrl+prophylactic",
    "sample_5": "TELO HAEC+SB+cGMP", "sample_6": "TELO HAEC+SB+cGMP+prophylactic",
    "sample_7": "TELO HAEC+SB+cAMP+cGMP", "sample_8": "TELO HAEC+SB+cAMP+cGMP+prophylactic",
    "sample_18": "TELO Day0", "sample_10": "iPSC EC+VEGF (10ng/mL)",
    "sample_11": "iPSC EC+VEGF (10ng/mL)+SB", "sample_12": "iPSC EC+VEGF (10ng/mL)+SB+cAMP",
    "sample_13": "iPSC EC+VEGF (10ng/mL)+SB+cGMP",
    "sample_14": "iPSC EC+VEGF (10ng/mL)+SB+cAMP+cGMP",
    "sample_15": "iPSC EC+VEGF (10ng/mL)+SB+cAMP+cGMP+prophylactic",
    "sample_16": "iPSC EC_day40(ctrl)", "sample_17": "iPSC EC(ctrl)+prophylactic",
    "sample_9": "iPSC EC_day0 (ctrl)",
}

data = pd.read_csv(ALL_CSV, index_col=0)
for i in range(1, 19):
    if f"sample_{i}" != "sample_9":
        for r in range(3):
            del data[f"sample_{i}-{r}"]
for c in ["37-39 cAMP low-D60", "43-45 GMP-D60", "55 SB low-D60", "56 SB low-D60",
          "HAEC-p3-young", "HAEC-D40", "Telo-HAEC-young", "Telo-HAEC-D40",
          "HAEC-VEGF-30ng/D40", "HAEC-FBS-3%/D40"]:
    del data[c]

col = []
for i in data.columns:
    if i != "gene_official" and "HAEC" not in i:
        s = i.split("-")[0]
        col.append(i.replace(s, batch2[s]))
    else:
        col.append(i)
data.columns = col

ens_gene = data.index
data.index = data["gene_official"]
del data["gene_official"]

adata2 = ad.AnnData(data.T)
adata2.var["gene_id"] = ens_gene
adata2.var_names_make_unique()
adata2

In [ ]:
# batch 3 matrix: keep the two day-60 conditions compared in this panel
batch3 = {
    "sample1": "iPSC-EC 5percent FBS gfp DAY60",
    "sample2": "iPSC-EC 5percent FBS circ DAY60",
    "sample3": "iPSC-EC VEGF+SB+3percent FBS DAY60",
    "sample4": "iPSC-EC 3percent FBS DAY60",
}

d3 = pd.read_csv(D60_CSV, index_col=0)
ens3 = d3.index
d3.index = d3["gene_official"]
del d3["gene_official"]

for i in range(1, 5):
    if f"sample_{i}" not in ("sample_3", "sample_1"):
        for r in range(3):
            del d3[f"sample{i}-{r}"]

col = []
for i in d3.columns:
    if i != "gene_official" and "sample" in i:
        s = i.split("-")[0]
        col.append(i.replace(s, batch3[s]))
    else:
        col.append(i)
d3.columns = col

adata3 = ad.AnnData(d3.T)
adata3.var["gene_id"] = ens3
adata3.var_names_make_unique()
adata3

In [ ]:
adata = ad.concat([adata2, adata3], axis=0)
adata.obs["samples_rep"] = pd.Categorical(adata.obs_names)
adata.obs["samples"] = pd.Categorical(
    [i if "HAEC-" in i else i[:-2] for i in adata.obs_names])

sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

hvg = pd.read_csv(HVG_CSV, index_col=1)
adata = adata[:, [g for g in hvg.index if g in adata.var.index]].copy()
print(adata.shape, "samples x highly-variable genes")

In [ ]:
cell_markers = {
    "Endothelial Cells": ["FABP4", "GPIHBP1", "CD34", "PECAM1", "VWF", "VEGFA"],
    "Fibroblasts": ["SERPINF1", "RGS5", "MMP7", "LUM", "BGN", "PDGFRA"],
}
filtered_cell_markers = {ct: [g for g in ms if g in adata.var.index]
                         for ct, ms in cell_markers.items()}

adata_non0 = adata[adata.obs["samples"] != "iPSC EC_day0 (ctrl)"].copy()
adata_non0.obs["samples"] = adata_non0.obs["samples"].cat.remove_unused_categories()

genes = sum(filtered_cell_markers.values(), [])
SHORT = {c: c.replace(" DAY60", "").replace("iPSC-EC ", "").replace("percent", "%")
         for c in adata_non0.obs["samples"].cat.categories}
order = [SHORT[c] for c in adata_non0.obs["samples"].cat.categories]
PALETTE = {order[0]: "#8d8dc6", order[1]: "#7cc4a4"}   # periwinkle / mint

df = adata_non0[:, genes].to_df()
df["group"] = [SHORT[s] for s in adata_non0.obs["samples"]]
df.groupby("group").size()

In [ ]:
def stars(p):
    if p < 1e-4: return "****"
    if p < 1e-3: return "***"
    if p < 1e-2: return "**"
    if p < 0.05: return "*"
    return "ns"


np.random.seed(RNG)   # sns.stripplot jitters the replicate dots with the global RNG

nrow, ncol = 2, 6
fig, axes = plt.subplots(nrow, ncol, figsize=(17, 8))
rows = []

for ax, gene in zip(axes.flat, genes):
    sns.violinplot(data=df, x="group", y=gene, order=order, hue="group",
                   hue_order=order, palette=PALETTE, legend=False,
                   cut=0, inner=None, linewidth=0.8, ax=ax)
    sns.boxplot(data=df, x="group", y=gene, order=order, width=0.13,
                showcaps=False, showfliers=False, boxprops={"facecolor": "black"},
                medianprops={"color": "white", "linewidth": 1.2},
                whiskerprops={"color": "black", "linewidth": 1.2}, ax=ax)
    sns.stripplot(data=df, x="group", y=gene, order=order, color="black",
                  size=5, jitter=0.12, alpha=0.9, ax=ax)

    a = df.loc[df["group"] == order[0], gene]
    b = df.loc[df["group"] == order[1], gene]
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    rows.append({"gene": gene, f"mean_{order[0]}": a.mean(),
                 f"mean_{order[1]}": b.mean(), "n_per_group": len(a),
                 "welch_p": p, "significance": stars(p)})

    lo, hi = df[gene].min(), df[gene].max()
    span = (hi - lo) or 1.0
    if SHOW_STATS:
        y, h = hi + 0.14 * span, 0.05 * span
        ax.plot([0, 0, 1, 1], [y, y + h, y + h, y], lw=1.1, c="0.35")
        ax.text(0.5, y + h * 1.15, stars(p), ha="center", va="bottom",
                fontsize=12, fontweight="bold", color="0.15")
        ax.text(0.5, y + h * 0.05, f"p={p:.3g}", ha="center", va="top",
                fontsize=8, color="0.45")
        ax.set_ylim(lo - 0.08 * span, y + h * 5.5)
    else:
        ax.set_ylim(lo - 0.10 * span, hi + 0.10 * span)

    ax.set_title(gene, fontstyle="italic", fontsize=13, pad=8)
    ax.set_xlabel(""); ax.set_ylabel("")
    sns.despine(ax=ax)

for r in range(nrow):
    axes[r, 0].set_ylabel("log-norm expr")
    for c in range(ncol):
        if r < nrow - 1:
            axes[r, c].set_xticklabels([])
        else:
            axes[r, c].set_xticklabels(order, rotation=20, ha="right")

for r, label in enumerate(filtered_cell_markers):
    box = axes[r, 0].get_position()
    fig.text(0.005, (box.y0 + box.y1) / 2, label, rotation=90,
             va="center", ha="left", fontsize=12, fontweight="bold", color="0.25")

fig.suptitle("Endothelial vs fibroblast markers — iPSC-EC Day 60 conditions",
             fontsize=14, fontweight="bold")
fig.tight_layout(rect=[0.025, 0, 1, 0.96])

stem = "iPSC_EC_Day60_marker_violin"
fig.savefig(OUT / f"{stem}.png", dpi=300, bbox_inches="tight")
fig.savefig(OUT / f"{stem}.pdf", bbox_inches="tight")
pd.DataFrame(rows).to_csv(OUT / f"{stem}_stats.csv", index=False)
pd.DataFrame(rows)